# V7 CertGen CIFAR-10 Generation T4x2 Bookrun


Runtime estimates are planning-only. Actual timing logs are `run_log_only`. `claim_allowed=false` for every cell output. RESUME must stay enabled for long runs. Copy-back instructions are included at the end. This notebook writes status JSON and output ZIP artifacts only; it does not run certificates or paper result generation.


Expected platform: Kaggle. Accelerator: T4x2. Internet/model access may be required. Required checkpoints: `google/ddpm-cifar10-32`, `FrankCCCCC/ddpm_ema_cifar10`, `FrankCCCCC/cfm-cifar10-32`. Required output ZIP: `/kaggle/working/certgen_cifar10_generation_outputs.zip`. Estimated runtime: 1k/model 10-60 min/model; 10k/model 1-8 hr/model; 50k/model 6-24+ hr/model.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
SAMPLE_COUNT_PER_MODEL = int(os.environ.get('SAMPLE_COUNT_PER_MODEL', '1000'))
RESUME = True
STOP_ON_FIRST_MODEL_FAILURE = True
WRITE_BLOCKED_STATUS_ON_FAILURE = True
CLAIM_ALLOWED = False
EVIDENCE_STATUS = 'run_log_only'
GPU_SHARDS = {0: [0, 499], 1: [500, 999]}
CHECKPOINTS = ['google/ddpm-cifar10-32', 'FrankCCCCC/ddpm_ema_cifar10', 'FrankCCCCC/cfm-cifar10-32']
print({'sample_count': SAMPLE_COUNT_PER_MODEL, 'resume': RESUME, 'claim_allowed': CLAIM_ALLOWED})

In [ ]:
print('input discovery')
print('kaggle input exists', Path('/kaggle/input').exists())
print('working free gb', round(shutil.disk_usage('/kaggle/working').free / 1e9, 2) if Path('/kaggle/working').exists() else 'not_kaggle')

In [ ]:
# Package checksum display and dependency install placeholder.
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'accelerate', 'torchvision'], check=True)
print('dependency install cell complete or skipped')

In [ ]:
try:
    import torch
    print('cuda available', torch.cuda.is_available(), 'device count', torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        print(idx, torch.cuda.get_device_name(idx))
except Exception as exc:
    print('torch unavailable before install', exc)

In [ ]:
# Generation config validation and per-model dry-run load check placeholder.
status = {'claim_allowed': False, 'evidence_status': 'run_log_only', 'generation_status': 'planned_or_running', 'checkpoints': CHECKPOINTS, 'gpu_shards': GPU_SHARDS}
Path('/kaggle/working/generation_status.json').write_text(json.dumps(status, indent=2), encoding='utf-8')
print('generation_status.json written')

In [ ]:
# Per-model two-GPU generation template. No NCCL, no distributed training.
# GPU 0 seeds 0-499, GPU 1 seeds 500-999.
# On checkpoint failure: write generation_blocked_status.json and stop; do not mark partial success complete.
for checkpoint in CHECKPOINTS:
    print('planned generation checkpoint', checkpoint)
    for shard_id, seed_range in GPU_SHARDS.items():
        print('planned shard', shard_id, seed_range)

In [ ]:
# Manifest merge, duplicate seed/path/hash validation, sample image grid preview saved as run_log_only.
out_dir = Path('/kaggle/working/certgen_generation_outputs')
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'manifest.json').write_text(json.dumps({'claim_allowed': False, 'evidence_status': 'run_log_only', 'status': 'placeholder_until_real_run'}, indent=2), encoding='utf-8')
(out_dir / 'timing_summary.json').write_text(json.dumps({'run_log_only': True}, indent=2), encoding='utf-8')

In [ ]:
zip_path = Path('/kaggle/working/certgen_cifar10_generation_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path('/kaggle/working/certgen_generation_outputs').rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to('/kaggle/working'))
print('output ZIP', zip_path)
print('copy-back: download this zip and run commands/v7_cpu_execution/04_import_generation_output_zip.sh')